In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../Data/songs_data/data.csv")

In [3]:
df.head()

,artist,song,link,text
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \r\nA..."
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \r\nTouch me gen..."
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \r\nWhy I had...
3,ABBA,Bang,/a/abba/bang_20598415.html,Making somebody happy is a question of give an...
4,ABBA,Bang-A-Boomerang,/a/abba/bang+a+boomerang_20002668.html,Making somebody happy is a question of give an...


In [4]:
df.shape

(57650, 4)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57650 entries, 0 to 57649
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   artist  57650 non-null  object
 1   song    57650 non-null  object
 2   link    57650 non-null  object
 3   text    57650 non-null  object
dtypes: object(4)
memory usage: 1.8+ MB


In [6]:
df.isnull().sum()

artist    0
song      0
link      0
text      0
dtype: int64

In [7]:
df.drop(columns='link',inplace=True)

In [8]:
df.head()

,artist,song,text
0,ABBA,Ahe's My Kind Of Girl,"Look at her face, it's a wonderful face \r\nA..."
1,ABBA,"Andante, Andante","Take it easy with me, please \r\nTouch me gen..."
2,ABBA,As Good As New,I'll never know why I had to go \r\nWhy I had...
3,ABBA,Bang,Making somebody happy is a question of give an...
4,ABBA,Bang-A-Boomerang,Making somebody happy is a question of give an...


In [9]:
df['text'][0]

"Look at her face, it's a wonderful face  \r\nAnd it means something special to me  \r\nLook at the way that she smiles when she sees me  \r\nHow lucky can one fellow be?  \r\n  \r\nShe's just my kind of girl, she makes me feel fine  \r\nWho could ever believe that she could be mine?  \r\nShe's just my kind of girl, without her I'm blue  \r\nAnd if she ever leaves me what could I do, what could I do?  \r\n  \r\nAnd when we go for a walk in the park  \r\nAnd she holds me and squeezes my hand  \r\nWe'll go on walking for hours and talking  \r\nAbout all the things that we plan  \r\n  \r\nShe's just my kind of girl, she makes me feel fine  \r\nWho could ever believe that she could be mine?  \r\nShe's just my kind of girl, without her I'm blue  \r\nAnd if she ever leaves me what could I do, what could I do?\r\n\r\n"

In [10]:
import nltk

In [11]:
from nltk.stem.porter import  PorterStemmer
ps = PorterStemmer()

In [12]:
def stem(text):
    y=[]
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [13]:
df['text'][0]

"Look at her face, it's a wonderful face  \r\nAnd it means something special to me  \r\nLook at the way that she smiles when she sees me  \r\nHow lucky can one fellow be?  \r\n  \r\nShe's just my kind of girl, she makes me feel fine  \r\nWho could ever believe that she could be mine?  \r\nShe's just my kind of girl, without her I'm blue  \r\nAnd if she ever leaves me what could I do, what could I do?  \r\n  \r\nAnd when we go for a walk in the park  \r\nAnd she holds me and squeezes my hand  \r\nWe'll go on walking for hours and talking  \r\nAbout all the things that we plan  \r\n  \r\nShe's just my kind of girl, she makes me feel fine  \r\nWho could ever believe that she could be mine?  \r\nShe's just my kind of girl, without her I'm blue  \r\nAnd if she ever leaves me what could I do, what could I do?\r\n\r\n"

In [42]:
 df = df.sample(10000, random_state=42).reset_index(drop=True)

In [43]:
df['text'] = df['text'].apply(stem)

In [44]:
df.sample(5)

,artist,song,text
5249,John Martyn,CoolTide,"so cool, what a cool time it' so cool, what a ..."
728,Nick Drake,Fruit Tree,fame is but a fruit tree so veri unsound. it c...
3171,David Bowie,Leon Takes Us Outside,"valentin day, 25, june, 16th, wednesday, juli ..."
4253,INXS,The Strangest Party (These Are The Times),welcom to the strangest parti babi it' like we...
8753,Queens Of The Stone Age,Era Vulgaris,i play a game 'til i'm dead or on a magazin i ...


In [45]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [46]:
tfid = TfidfVectorizer(analyzer='word',stop_words='english');

In [47]:
matrix=tfid.fit_transform(df['text'])

In [48]:
similer = cosine_similarity(matrix)

In [49]:
similer[0]

array([1.        , 0.00986595, 0.00200102, ..., 0.01913919, 0.00995377,
       0.00592196], shape=(10000,))

In [50]:
df[df['song']=='Lady Whiskey'].index[0]

np.int64(9286)

## Recommender Function

In [51]:
def recommender(song_name):
    if song_name not in df['song'].values:
        print("Sorry! Song not Found...")
        return
    song_index=df[df['song']==song_name].index[0]
    distances = similer[song_index]
    song_list = sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:9]
    recommendations = []
    for i in song_list:
        recommendations.append({
            "song": df.iloc[i[0]].song,
            "match": min(round((i[1] * 100) + 50, 2), 99)
        })
    return recommendations
    

In [52]:
recommender('Innocent Bystander')

[{'song': "Goin' Out Of My Head", 'match': np.float64(93.91)},
 {'song': 'Just Keep Thinking About You', 'match': np.float64(83.93)},
 {'song': "I Don't Think You Know Me", 'match': np.float64(83.75)},
 {'song': 'Another Year', 'match': np.float64(83.54)},
 {'song': 'He Thinks I Still Care', 'match': np.float64(82.37)},
 {'song': "Can't Stop Thinking About You", 'match': np.float64(81.67)},
 {'song': 'Heart Over Head', 'match': np.float64(81.49)},
 {'song': 'High Cool', 'match': np.float64(80.0)}]

In [53]:
import pickle

In [54]:
pickle.dump(df, open('song_list.pkl', 'wb'))
pickle.dump(similer, open('similer.pkl', 'wb'))